In [12]:
import numpy as np
import pymc3 as pm
from dataclasses import dataclass, asdict,
from collections import Counter


In [2]:


@dataclass(eq=True, frozen=True)
class Arm:
    color: str
    theme: str


@dataclass
class Response:
    arm: Arm
    reward: int


class Env(object):
    arms = [
        Arm('red', 'art'),
        Arm('red', 'food'),
        Arm('blue', 'art'),
        Arm('blue', 'food')
    ]

    phis = np.array([
        [0, 0, 1],
        [0, 1, 1],
        [1, 0, 1],
        [1, 1, 1]
    ]).T


class User(object):
    def __init__(self):
        # color * 0.2 + theme * 0.8 - 4
        self.preferences = {}

        for arm in Env.arms:
            color = 0 if arm.color == 'red' else 1
            theme = 0 if arm.theme == 'art' else 1
            score = color * 0.2 + theme * 0.8 - 2
            self.preferences[arm] = 1 / (1 + np.exp(-score))

    def response(self, arm: Arm):
        pref = self.preferences[arm]
        return Response(arm, 1) if np.random.random() < pref else Response(arm, 0)


class StateRepository:
    def __init__(self, responses=[]):
        self.responses = responses

    @property
    def cum_rewards(self):
        rewards = []
        for arm in Env.arms:
            rewards.append(sum([res.reward for res in self.responses if res.arm == arm]))
        return rewards

    @property
    def counts(self):
        c = Counter()
        c.update({arm: 0 for arm in Env.arms})
        arms = map(lambda res: res.arm, self.responses)
        c.update(arms)
        return [c[arm] for arm in Env.arms]

    def append(self, response: Response):
        self.responses.append(response)



In [3]:
class BerTSAgent:
    def get_arm(self, counts, wins):
        if 0 in counts:
            return Env.arms[counts.index(0)]

        model = pm.Model()
        with model:
            beta = pm.Normal('beta', mu=0, sigma=10, shape=3)
            linpred = pm.math.dot(beta, Env.phis)
            theta = pm.Deterministic('theta', 1 / (1 + pm.math.exp(-linpred)))
            obs = pm.Binomial('obs', n=counts, p=theta, observed=wins)
            trace = pm.sample(1000, chains=1)

        sample = pm.sample_posterior_predictive(trace, samples=1, model=model, var_names=['theta'])
        idx = np.argmax(sample['theta'])
        return Env.arms[idx]

In [4]:
user = User()
repo = StateRepository()
agent = BerTSAgent()

In [6]:
N = 5
for i in range(N):
    # ai = np.random.randint(0, 4)
    # arm = Env.arms[ai]
    arm = agent.get_arm(repo.counts, repo.cum_rewards)
    res = user.response(arm)
    repo.append(res)

/var/folders/4_/vrr8kzqn5b9dxsprxn13022m0000gn/T/ipykernel_28890/1085047266.py:12: FutureWarning: In v4.0, pm.sample will return an `arviz.InferenceData` object instead of a `MultiTrace` by default. You can pass return_inferencedata=True or return_inferencedata=False to be safe and silence this warning.
  trace = pm.sample(1000, chains=1)
Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (1 chains in 1 job)
NUTS: [beta]


/Users/kotarohara/repo/Python/web-optimization/venv/lib/python3.9/site-packages/scipy/stats/_continuous_distns.py:624: RuntimeWarning: overflow encountered in _beta_ppf
  return _boost._beta_ppf(q, a, b)
Sampling 1 chain for 1_000 tune and 1_000 draw iterations (1_000 + 1_000 draws total) took 1 seconds.
Only one chain was sampled, this makes it impossible to run some convergence checks
/Users/kotarohara/repo/Python/web-optimization/venv/lib/python3.9/site-packages/pymc3/sampling.py:1689: UserWarning: samples parameter is smaller than nchains times ndraws, some draws and/or chains may not be represented in the returned posterior predictive sample
  warnings.warn(


/var/folders/4_/vrr8kzqn5b9dxsprxn13022m0000gn/T/ipykernel_28890/1085047266.py:12: FutureWarning: In v4.0, pm.sample will return an `arviz.InferenceData` object instead of a `MultiTrace` by default. You can pass return_inferencedata=True or return_inferencedata=False to be safe and silence this warning.
  trace = pm.sample(1000, chains=1)
Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (1 chains in 1 job)
NUTS: [beta]


/Users/kotarohara/repo/Python/web-optimization/venv/lib/python3.9/site-packages/scipy/stats/_continuous_distns.py:624: RuntimeWarning: overflow encountered in _beta_ppf
  return _boost._beta_ppf(q, a, b)
Sampling 1 chain for 1_000 tune and 1_000 draw iterations (1_000 + 1_000 draws total) took 1 seconds.
Only one chain was sampled, this makes it impossible to run some convergence checks
/Users/kotarohara/repo/Python/web-optimization/venv/lib/python3.9/site-packages/pymc3/sampling.py:1689: UserWarning: samples parameter is smaller than nchains times ndraws, some draws and/or chains may not be represented in the returned posterior predictive sample
  warnings.warn(


In [7]:
repo.cum_rewards

[0, 1, 0, 0]

In [8]:
repo.counts

[1, 3, 1, 1]

In [13]:
asdict(repo.responses[0].arm)

{'color': 'red', 'theme': 'art'}